# Proyecto Rain in Australia
## Fase 1: Definición del problema y entorno reproducible

**Asignatura:** MCDI501 – Estadística Computacional

**Integrantes:**
- Enzo Pinilla
- Claudio Alarcón
- Luis Espinoza

**Docente:** Jean Paul Maidana

**Objetivo de la Fase 1:** 

# Problemática

Nombre: Rain in Australia (Lluvia en Australia) Tipo: Clasificación binaria
Variable a Predecir: ¿Lloverá mañana? (Sí / No)
Descripción:
Contiene aproximadamente 10 años de observaciones meteorológicas diarias de numerosas esta-
ciones climáticas en Australia. El objetivo es predecir si lloverá al día siguiente basándose en las
observaciones del día actual.
Variables: temperatura mínima y máxima, cantidad de lluvia, velocidad y dirección del viento,
humedad a las 9am y 3pm, presión atmosférica, nubosidad, ubicación de la estación.
Aplicaciones prácticas: predicción meteorológica, planificación agrícola, gestión de recursos
hídricos, turismo y eventos al aire libre.
Fuente: https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package/d
ata
Desafíos: datos faltantes en varias variables numéricas, posible estacionalidad, diferencias entre
ubicaciones geográficas, tamaño grande (el remuestreo del modelo puede ser más lento)

In [34]:
import pandas as pd
import numpy as np
from pathlib import Path

# Función de carga de datos

Para favorecer la reutilización del código y mantener una estructura reproducible, la lectura del dataset se implementa mediante una función.

In [35]:
def cargar_datos(ruta):
    """
    Carga el dataset desde la ruta especificada.

    Parameters
    ----------
    ruta : str
        Ruta del archivo CSV.

    Returns
    -------
    pandas.DataFrame
        DataFrame con los datos cargados.
    """
    ruta = Path(ruta)
    rutas_posibles = [
        ruta,
        Path.cwd() / ruta,
        Path.cwd().parent / ruta,
        Path.cwd() / "data" / "raw" / "weatherAUS.csv",
        Path.cwd().parent / "data" / "raw" / "weatherAUS.csv",
    ]

    for ruta_posible in rutas_posibles:
        if ruta_posible.exists():
            return pd.read_csv(ruta_posible)

    raise FileNotFoundError(
        "No se encontro el CSV. Usa data/raw/Fifa_world_cup_matches.csv "
        "o verifica el directorio desde donde se ejecuta el notebook."
    )

# Carga inicial del dataset

La siguiente celda realiza la lectura del archivo CSV almacenado en la carpeta data/raw.

In [36]:
df = cargar_datos("../data/raw/weatherAUS.csv")

# Validación de estructura

A continuación se verifica la cantidad de registros y variables disponibles en el dataset.

In [37]:
print("Dimensiones del dataset:")
print(df.shape)

Dimensiones del dataset:
(145460, 23)


# Vista preliminar de los datos

Se muestran las primeras filas para comprender la estructura del conjunto de datos y validar que la carga fue exitosa.

In [38]:
df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


# Información general del dataset

La siguiente celda permite inspeccionar tipos de datos y posibles valores faltantes.

In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           145460 non-null  str    
 1   Location       145460 non-null  str    
 2   MinTemp        143975 non-null  float64
 3   MaxTemp        144199 non-null  float64
 4   Rainfall       142199 non-null  float64
 5   Evaporation    82670 non-null   float64
 6   Sunshine       75625 non-null   float64
 7   WindGustDir    135134 non-null  str    
 8   WindGustSpeed  135197 non-null  float64
 9   WindDir9am     134894 non-null  str    
 10  WindDir3pm     141232 non-null  str    
 11  WindSpeed9am   143693 non-null  float64
 12  WindSpeed3pm   142398 non-null  float64
 13  Humidity9am    142806 non-null  float64
 14  Humidity3pm    140953 non-null  float64
 15  Pressure9am    130395 non-null  float64
 16  Pressure3pm    130432 non-null  float64
 17  Cloud9am       89572 non-null   float64


# Estadísticas descriptivas iniciales

Se generan métricas descriptivas básicas para obtener una primera comprensión del contenido del dataset.

In [40]:
df.describe()

,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm
count,143975.000000,144199.000000,142199.000000,82670.000000,75625.000000,135197.000000,143693.000000,142398.000000,142806.000000,140953.000000,130395.00000,130432.000000,89572.000000,86102.000000,143693.000000,141851.00000
mean,12.194034,23.221348,2.360918,5.468232,7.611178,40.035230,14.043426,18.662657,68.880831,51.539116,1017.64994,1015.255889,4.447461,4.509930,16.990631,21.68339
std,6.398495,7.119049,8.478060,4.193704,3.785483,13.607062,8.915375,8.809800,19.029164,20.795902,7.10653,7.037414,2.887159,2.720357,6.488753,6.93665
min,-8.500000,-4.800000,0.000000,0.000000,0.000000,6.000000,0.000000,0.000000,0.000000,0.000000,980.50000,977.100000,0.000000,0.000000,-7.200000,-5.40000
25%,7.600000,17.900000,0.000000,2.600000,4.800000,31.000000,7.000000,13.000000,57.000000,37.000000,1012.90000,1010.400000,1.000000,2.000000,12.300000,16.60000
50%,12.000000,22.600000,0.000000,4.800000,8.400000,39.000000,13.000000,19.000000,70.000000,52.000000,1017.60000,1015.200000,5.000000,5.000000,16.700000,21.10000
75%,16.900000,28.200000,0.800000,7.400000,10.600000,48.000000,19.000000,24.000000,83.000000,66.000000,1022.40000,1020.000000,7.000000,7.000000,21.600000,26.40000
max,33.900000,48.100000,371.000000,145.000000,14.500000,135.000000,130.000000,87.000000,100.000000,100.000000,1041.00000,1039.600000,9.000000,9.000000,40.200000,46.70000


Calculo de la Media, Mediana y MOda de la columna MinTemp (Medidas de tendencia central)

In [65]:
MinTemp = df.iloc[1:, 2].tolist()
media_min_temp = df["MinTemp"].mean()
print(f"Media MinTemp: {media_min_temp:.2f}")

media_min_temp = df["MinTemp"].mean()
mediana_min_temp = df["MinTemp"].median()

print(f"Mediana MinTemp: {mediana_min_temp:.2f}")

moda_min_temp = df["MinTemp"].mode()[0]

print(f"Moda MinTemp: {moda_min_temp:.2f}")

diff_pct = abs(media_min_temp - mediana_min_temp) / mediana_min_temp * 100
print(f"\nDiferencia media - mediana: {diff_pct:.1f}%")

#Faltantes
print(f"Cantidad de NaN: {df['MinTemp'].isna().sum()}")

Media MinTemp: 12.19
Mediana MinTemp: 12.00
Moda MinTemp: 11.00

Diferencia media - mediana: 1.6%
Cantidad de NaN: 1485


Medidas de Dispersion

In [74]:
#Rango
rango = df["MinTemp"].max() - df["MinTemp"].min()
print(f"Rango: {rango}")

# IQR
Q1 = df["MinTemp"].quantile(0.25)
Q3 = df["MinTemp"].quantile(0.75)
IQR = Q3 - Q1
print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")

# Varianza: especificar ddof siempre
# ddof=0 población (divide por n)
# ddof=1 muestra (divide por n-1) uso estándar en análisis
var = df["MinTemp"].var(ddof=0)
var_muestra = df["MinTemp"].var(ddof=1)
print(f"\nVarianza (ddof=0): {var:.2f}")
print(f"Varianza muestral (ddof=1): {var_muestra:.2f}")
print(f"Diferencia: {var_muestra - var:.2f}")

# Desviación estándar muestral
std_muestra = df["MinTemp"].std(ddof=1)
print(f"\nDesviación estándar (muestra): {std_muestra:.2f}")

# Coeficiente de variación
media = df["MinTemp"].mean()
CV = (std_muestra / media) * 100
print(f"Media: {media:.2f}")
print(f"CV: {CV:.2f}%")

Rango: 42.4
Q1: 7.6, Q3: 16.9, IQR: 9.299999999999999

Varianza (ddof=0): 40.94
Varianza muestral (ddof=1): 40.94
Diferencia: 0.00

Desviación estándar (muestra): 6.40
Media: 12.19
CV: 52.47%


# Interpretación inicial

La carga del dataset fue realizada correctamente y se verificó la disponibilidad de las variables necesarias para el análisis.

La inspección inicial permite conocer la cantidad de registros, la estructura de las columnas y los tipos de datos presentes en el conjunto de información.

Esta revisión preliminar constituye el punto de partida para las actividades de exploración y limpieza que serán desarrolladas en las siguientes fases del proyecto.

# Conclusiones de la Fase 1

Durante esta primera fase se definió la problemática del proyecto, se establecieron los objetivos generales y específicos, se configuró la estructura inicial del repositorio y se validó la carga del dataset FIFA World Cup 2022.

Asimismo, se verificó el correcto funcionamiento del entorno reproducible basado en Python, Jupyter Notebook y GitHub, permitiendo garantizar que los análisis posteriores puedan ejecutarse de forma consistente y trazable.

Las siguientes etapas del proyecto estarán orientadas a la exploración de datos, la limpieza y transformación del dataset, el análisis descriptivo de las variables y la identificación de patrones asociados a la victoria de las selecciones participantes.